# 📡 Telecom ABSA — Evaluation Summary

**Aspect-Based Sentiment Analysis for Telecom Customer Feedback**

This notebook consolidates all evaluation findings from the model comparison, error analysis, and pattern analysis into a clean, presentable format.

---

**Models Compared:**
- Model 1: Logistic Regression + TF-IDF
- Model 2: DistilBERT fine-tuned

**Dataset:** 1,000 telecom customer feedback samples (15 aspects, 3 sentiments)

**Evaluation Date:** June 2026

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 1 — Imports and Setup
# ═══════════════════════════════════════════════════════════════

import json
import os
from collections import Counter
from IPython.display import display, Markdown, Image

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px

# Paths
MODELS_DIR = 'outputs/models'
EDA_DIR = 'outputs/eda'
DATA_DIR = 'data'

def load_json(path):
    if os.path.exists(path):
        with open(path) as f:
            return json.load(f)
    print(f'⚠️ Not found: {path}')
    return {}

print('✅ Imports complete')

---
## 📊 Dataset Overview

Summary of the test set used for all evaluations.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 2 — Dataset Overview
# ═══════════════════════════════════════════════════════════════

test_df = pd.read_csv(f'{DATA_DIR}/test.csv')
test_df['aspects'] = test_df['aspects'].apply(json.loads)
test_df['aspect_sentiments'] = test_df['aspect_sentiments'].apply(json.loads)

print(f'Test set: {len(test_df)} samples')
print(f'Columns: {list(test_df.columns)}')
print()

# Aspect distribution
aspect_counts = Counter()
for aspects in test_df['aspects']:
    aspect_counts.update(aspects)

aspect_dist = pd.DataFrame(
    sorted(aspect_counts.items(), key=lambda x: -x[1]),
    columns=['Aspect', 'Count']
)
aspect_dist['Percentage'] = (aspect_dist['Count'] / len(test_df) * 100).round(1)

print('Aspect Distribution in Test Set:')
display(aspect_dist.style.bar(subset=['Count'], color='#3498db'))

---
## 📈 Model Comparison Tables

Side-by-side performance comparison across all metrics.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 3 — Metrics Comparison Tables
# ═══════════════════════════════════════════════════════════════

m1 = load_json(f'{MODELS_DIR}/model1_metrics.json')
m2 = load_json(f'{MODELS_DIR}/model2_metrics.json')

def sg(d, *keys, default=0.0):
    """Safe get for nested dicts."""
    c = d
    for k in keys:
        c = c.get(k, default) if isinstance(c, dict) else default
    return c if c is not None else default

# Table 1: Aspect Detection
print('\n📋 TABLE 1 — Aspect Detection Performance')
print('=' * 60)

aspect_table = []
for split in ['train', 'val', 'test']:
    aspect_table.append({
        'Split': split.capitalize(),
        'Metric': 'Micro-F1',
        'Model 1': sg(m1, 'aspect_detection', split, 'micro', 'f1'),
        'Model 2': sg(m2, 'aspect_detection', split, 'micro', 'f1'),
    })
    aspect_table.append({
        'Split': split.capitalize(),
        'Metric': 'Macro-F1',
        'Model 1': sg(m1, 'aspect_detection', split, 'macro', 'f1'),
        'Model 2': sg(m2, 'aspect_detection', split, 'macro', 'f1'),
    })
    aspect_table.append({
        'Split': split.capitalize(),
        'Metric': 'Hamming Loss',
        'Model 1': sg(m1, 'aspect_detection', split, 'hamming_loss'),
        'Model 2': sg(m2, 'aspect_detection', split, 'hamming_loss'),
    })

df_aspect = pd.DataFrame(aspect_table)
df_aspect['Delta'] = df_aspect['Model 2'] - df_aspect['Model 1']
display(df_aspect.style.format({'Model 1': '{:.4f}', 'Model 2': '{:.4f}', 'Delta': '{:+.4f}'}))

# Table 2: Sentiment Classification
print('\n📋 TABLE 2 — Sentiment Classification Performance')
print('=' * 60)

sent_table = []
for split in ['train', 'val', 'test']:
    sent_table.append({
        'Split': split.capitalize(),
        'Metric': 'Accuracy',
        'Model 1': sg(m1, 'sentiment_classification', split, '_overall', 'accuracy'),
        'Model 2': sg(m2, 'sentiment_classification', split, '_overall', 'accuracy'),
    })
    sent_table.append({
        'Split': split.capitalize(),
        'Metric': 'Macro-F1',
        'Model 1': sg(m1, 'sentiment_classification', split, '_overall', 'macro_f1'),
        'Model 2': sg(m2, 'sentiment_classification', split, '_overall', 'macro_f1'),
    })
    sent_table.append({
        'Split': split.capitalize(),
        'Metric': 'Weighted-F1',
        'Model 1': sg(m1, 'sentiment_classification', split, '_overall', 'weighted_f1'),
        'Model 2': sg(m2, 'sentiment_classification', split, '_overall', 'weighted_f1'),
    })

df_sent = pd.DataFrame(sent_table)
df_sent['Delta'] = df_sent['Model 2'] - df_sent['Model 1']
display(df_sent.style.format({'Model 1': '{:.4f}', 'Model 2': '{:.4f}', 'Delta': '{:+.4f}'}))

---
## 🎯 Per-Aspect F1 Comparison

Which aspects does each model handle best?

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 4 — Per-Aspect F1 Comparison Chart
# ═══════════════════════════════════════════════════════════════

img_path = f'{EDA_DIR}/per_aspect_f1_comparison.png'
if os.path.exists(img_path):
    display(Image(filename=img_path, width=900))
else:
    print(f'⚠️ Chart not found: {img_path}')
    print('Run: python -m src.per_aspect_comparison')

---
## 🔀 Confusion Matrix Overview

Top 3 most confused aspects — side-by-side Model 1 vs Model 2.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 5 — Confusion Matrix Comparison
# ═══════════════════════════════════════════════════════════════

img_path = f'{EDA_DIR}/top3_confused_aspects_comparison.png'
if os.path.exists(img_path):
    display(Image(filename=img_path, width=900))
else:
    print(f'⚠️ Chart not found: {img_path}')
    print('Run: python -m src.confusion_matrix_analysis')

---
## ❌ Error Analysis Summary

Breakdown of misclassification patterns across the test set.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 6 — Error Analysis Summary
# ═══════════════════════════════════════════════════════════════

error_report = load_json(f'{MODELS_DIR}/error_pattern_report.json')
misclass = load_json(f'{MODELS_DIR}/misclassifications.json')

if misclass:
    total = misclass.get('total_test_samples', 0)
    errors = misclass.get('total_misclassified', 0)
    print(f'Total test samples: {total}')
    print(f'Misclassified: {errors} ({errors*100/max(total,1):.1f}%)')
    print()
    
    # Error category pie chart
    cats = misclass.get('error_category_distribution', {})
    if cats:
        fig = px.pie(
            values=list(cats.values()),
            names=list(cats.keys()),
            title='Error Category Distribution',
            color_discrete_sequence=px.colors.qualitative.Set2
        )
        fig.update_layout(paper_bgcolor='rgba(0,0,0,0)')
        fig.show()
    
    # Top 5 misclassified aspects
    top5 = error_report.get('top5_failure_aspects', [])
    if top5:
        df_top5 = pd.DataFrame(top5)
        fig = px.bar(
            df_top5, x='aspect', y='rate',
            title='Top 5 Aspects by Error Rate',
            color='rate', color_continuous_scale='Reds'
        )
        fig.update_layout(paper_bgcolor='rgba(0,0,0,0)')
        fig.show()
    
    # 3 example misclassifications
    print('\n📝 Example Misclassifications:')
    print('=' * 70)
    records = misclass.get('records', [])
    examples = []
    for r in records[:3]:
        for ae in r.get('aspect_errors', [])[:1]:
            examples.append({
                'Feedback': r['feedback'][:80] + '...',
                'Aspect': ae.get('aspect', ''),
                'True': ae.get('true_sentiment', ''),
                'Predicted': ae.get('predicted_sentiment', ''),
                'Error Type': ae.get('error_type', ''),
            })
    if examples:
        display(pd.DataFrame(examples))
else:
    print('⚠️ Run: python -m src.error_analysis')

---
## 🔄 Sentiment Misclassification Patterns

Three key patterns identified in how the model confuses sentiments.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 7 — Sentiment Misclassification Patterns
# ═══════════════════════════════════════════════════════════════

patterns = error_report.get('sentiment_patterns', {})

if patterns:
    for pattern_name, info in patterns.items():
        display(Markdown(f'### Pattern: `{pattern_name}` ({info.get("count", 0)} cases)'))
        display(Markdown(f'**Explanation:** {info.get("explanation", "")}'))
        display(Markdown('**Examples:**'))
        for ex in info.get('examples', [])[:3]:
            display(Markdown(f'> *\"{ex}\"*'))
        display(Markdown('---'))
else:
    print('⚠️ No pattern data. Run: python -m src.error_analysis')

---
## 📊 Multi-Aspect Challenge Analysis

Does the model struggle more with feedback mentioning many aspects?

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 8 — Multi-Aspect Challenge
# ═══════════════════════════════════════════════════════════════

img_path = f'{EDA_DIR}/error_rate_vs_aspect_count.png'
if os.path.exists(img_path):
    display(Image(filename=img_path, width=700))
else:
    # Display from error report data
    rates = error_report.get('multi_aspect_error_rates', {})
    if rates:
        df_rates = pd.DataFrame([
            {'Aspects': k, 'Error Rate': v['rate'], 'Total': v['total'], 'Errors': v['errors']}
            for k, v in rates.items()
        ])
        display(df_rates)
    else:
        print('⚠️ Run: python -m src.error_analysis')

---
## ✅ Model Selection Summary

Formal justification for the production model choice.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 9 — Model Selection Justification
# ═══════════════════════════════════════════════════════════════

md_path = f'{MODELS_DIR}/model_selection_justification.md'
if os.path.exists(md_path):
    with open(md_path) as f:
        display(Markdown(f.read()))
else:
    print('⚠️ Run: python -m src.model_selection_doc')

---
## 🚀 Possible Improvements

Actionable recommendations based on error analysis findings.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 10 — Improvements Report
# ═══════════════════════════════════════════════════════════════

md_path = f'{MODELS_DIR}/improvements_report.md'
if os.path.exists(md_path):
    with open(md_path) as f:
        display(Markdown(f.read()))
else:
    print('⚠️ Run: python -m src.improvements_report')

---

## 📋 End of Evaluation Summary

**Key Takeaways:**

1. Model 1 (LR+TF-IDF) outperforms on this small dataset due to data efficiency
2. Model 2 (DistilBERT) has architectural advantages that scale with more data
3. Top error sources: neutral confusion, multi-aspect disambiguation, noisy input
4. Recommended next steps: more training data, ensemble approach, active learning

---
*Generated as part of the ABSA Telecom NLP project evaluation pipeline.*